In [2]:
import json
import os

class DataManager:
    """
    Gerencia o carregamento e acesso aos dados do dicionário a partir de um arquivo JSON.
    """
    def __init__(self, filepath="cambridge_dictionary_data.json"):
        """
        Inicializa o DataManager.

        Args:
            filepath (str): O caminho para o arquivo JSON do dicionário.
        """
        self.filepath = filepath
        self.data = self._load_data()

    def _load_data(self):
        """
        Carrega os dados do arquivo JSON.

        Returns:
            dict: Os dados do dicionário ou um dicionário vazio em caso de erro.
        """
        if not os.path.exists(self.filepath):
            print(f"Erro: Arquivo de dados '{self.filepath}' não encontrado.")
            return {}
        try:
            with open(self.filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except json.JSONDecodeError:
            print(f"Erro: Falha ao decodificar o arquivo JSON '{self.filepath}'.")
            return {}
        except Exception as e:
            print(f"Ocorreu um erro inesperado ao carregar os dados: {e}")
            return {}

    def buscar_palavra(self, palavra):
        """
        Busca uma palavra no dicionário (case-insensitive).

        Args:
            palavra (str): A palavra a ser buscada.

        Returns:
            dict or None: Os dados da palavra se encontrada, caso contrário None.
        """
        palavra_lower = palavra.lower()
        return self.data.get(palavra_lower)

if __name__ == '__main__':
    # Teste rápido do DataManager
    dm = DataManager()
    if dm.data:
        print("Dados carregados com sucesso!")
        
        palavra_teste = "story"
        resultado = dm.buscar_palavra(palavra_teste)
        if resultado:
            print(f"\nDados para '{palavra_teste}':")
            print(json.dumps(resultado, indent=2))
        else:
            print(f"\nPalavra '{palavra_teste}' não encontrada.")

        palavra_inexistente = "xyz123"
        resultado_inexistente = dm.buscar_palavra(palavra_inexistente)
        if not resultado_inexistente:
            print(f"Palavra '{palavra_inexistente}' corretamente não encontrada.")
    else:
        print("Falha ao carregar os dados.")

Dados carregados com sucesso!

Dados para 'story':
{
  "word": "story",
  "part_of_speech": "noun",
  "grammar": "[ C ]",
  "pronunciations": {
    "uk": {
      "ipa": "\u02c8st\u0254\u02d0.ri",
      "audio": "https://dictionary.cambridge.org/media/english/uk_pron/u/uks/uksto/ukstore008.mp3"
    },
    "us": {
      "ipa": "\u02c8st\u0254\u02d0r.i",
      "audio": "https://dictionary.cambridge.org/media/english/us_pron/s/sto/story/story.mp3"
    }
  },
  "senses": [
    {
      "cefr_level": "A2",
      "definition": "a description, either true or imagined, of a connected series of events: ",
      "examples": [
        "Will you read/tell me a story, daddy?",
        "Martha chose her favourite book of bedtime stories.",
        "He writes children's stories.",
        "I don't know if it's true but it's a good story (= entertaining to listen to although probably not true).",
        "She gave me her version of what had happened, but it would be interesting to hear his half/side of 

In [3]:
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
# from data_manager import DataManager
import webbrowser # Para abrir links e simular "tocar áudio"

class DicionarioApp:
    """
    Classe principal para a aplicação de dicionário com interface Tkinter.
    """
    AUDIO_SYMBOL = "🔊" # U+1F50A

    def __init__(self, master_root):
        """
        Inicializa a aplicação.

        Args:
            master_root (tk.Tk): A janela raiz da aplicação.
        """
        self.root = master_root
        self.root.title("Dicionário Elegante")
        self.root.geometry("800x700")

        self.data_manager = DataManager() # Carrega os dados ao instanciar

        if not self.data_manager.data:
            messagebox.showerror("Erro de Dados", 
                                 "Não foi possível carregar os dados do dicionário. "
                                 "Verifique o arquivo 'seu_arquivo_dicionario.json' e reinicie a aplicação.")
            self.root.destroy()
            return
            
        self._configurar_estilos_ttk()
        self._criar_widgets()
        self._configurar_tags_texto()

    def _configurar_estilos_ttk(self):
        """Configura estilos para widgets ttk para uma aparência mais moderna."""
        style = ttk.Style()
        style.theme_use('clam') # 'clam', 'alt', 'default', 'classic'

        style.configure("TFrame", background="#f0f0f0")
        style.configure("TLabel", background="#f0f0f0", font=("Arial", 10))
        style.configure("TButton", font=("Arial", 10, "bold"), padding=5)
        style.configure("Search.TEntry", padding=5, font=("Arial", 11))
        
        style.configure("Header.TLabel", font=("Arial", 12, "bold"), foreground="#333333")
        style.configure("WordTitle.TLabel", font=("Arial", 20, "bold"), foreground="#003366") # Exemplo, não usado diretamente no Text
        style.configure("POS.TLabel", font=("Arial", 14, "italic"), foreground="#555555") # Exemplo

    def _criar_widgets(self):
        """Cria e organiza os widgets da interface."""
        
        # Frame principal
        main_frame = ttk.Frame(self.root, padding="10 10 10 10")
        main_frame.pack(expand=True, fill=tk.BOTH)

        # --- Frame de Busca ---
        search_frame = ttk.Frame(main_frame, padding="5")
        search_frame.pack(fill=tk.X, pady=(0, 10))

        ttk.Label(search_frame, text="Digite a palavra:", font=("Arial", 11)).pack(side=tk.LEFT, padx=(0, 5))
        
        self.search_entry = ttk.Entry(search_frame, width=40, style="Search.TEntry")
        self.search_entry.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=5)
        self.search_entry.bind("<Return>", self._processar_busca_event)

        search_button = ttk.Button(search_frame, text="Buscar", command=self._processar_busca)
        search_button.pack(side=tk.LEFT, padx=(5, 0))

        # --- Frame de Resultados ---
        results_frame = ttk.Frame(main_frame, padding="5")
        results_frame.pack(expand=True, fill=tk.BOTH)
        
        # Usando scrolledtext.ScrolledText para rolagem automática
        self.results_text = scrolledtext.ScrolledText(
            results_frame, 
            wrap=tk.WORD, 
            font=("Arial", 11), 
            padx=10, pady=10,
            relief=tk.SOLID,
            bd=1,
            bg="white" # Fundo branco para a área de texto
        )
        self.results_text.pack(expand=True, fill=tk.BOTH)
        self.results_text.configure(state='disabled') # Inicia desabilitado para edição

    def _configurar_tags_texto(self):
        """Configura as tags de estilo para o widget Text."""
        self.results_text.tag_configure("word_title", font=("Arial", 22, "bold"), foreground="#003366", spacing1=5, spacing3=5)
        self.results_text.tag_configure("pos_grammar", font=("Arial", 14, "italic"), foreground="#444444", spacing3=3)
        
        self.results_text.tag_configure("heading", font=("Arial", 13, "bold", "underline"), foreground="#222222", spacing1=10, spacing3=5)
        
        self.results_text.tag_configure("pron_label", font=("Arial", 11, "bold"), foreground="#005588")
        self.results_text.tag_configure("ipa", font=("Arial", 11, "italic"), foreground="#333333")
        
        self.results_text.tag_configure("audio_link", foreground="blue", underline=True)
        self.results_text.tag_bind("audio_link", "<Button-1>", self._clique_audio_link)
        
        self.results_text.tag_configure("cefr_level", font=("Arial", 11, "bold"), foreground="#880000", spacing3=3)
        self.results_text.tag_configure("definition", font=("Arial", 11), foreground="#111111", spacing3=3, lmargin1=10, lmargin2=10)
        self.results_text.tag_configure("example_intro", font=("Arial", 10, "italic"), foreground="#555555", lmargin1=20, lmargin2=20)
        self.results_text.tag_configure("example_text", font=("Arial", 10), foreground="#333333", lmargin1=25, lmargin2=25, spacing3=2)

        self.results_text.tag_configure("smart_topic_name", font=("Arial", 11, "bold"), foreground="#006622", lmargin1=10, lmargin2=10)
        self.results_text.tag_configure("smart_related_word", font=("Arial", 10), foreground="#006622", lmargin1=20, lmargin2=20)
        self.results_text.tag_configure("smart_link", foreground="blue", underline=True, lmargin1=20, lmargin2=20)
        self.results_text.tag_bind("smart_link", "<Button-1>", self._clique_smart_link)

        self.results_text.tag_configure("error", font=("Arial", 12, "bold"), foreground="red", justify=tk.CENTER, spacing1=20)
        self.results_text.tag_configure("info", font=("Arial", 11, "italic"), foreground="gray", justify=tk.CENTER, spacing1=20)


    def _processar_busca_event(self, event=None): # Adicionado event=None
        """Processa o evento de busca (ex: Enter no Entry)."""
        self._processar_busca()

    def _processar_busca(self):
        """Obtém a palavra e exibe os resultados."""
        palavra = self.search_entry.get().strip()
        if not palavra:
            self._exibir_mensagem("Por favor, digite uma palavra.", "info")
            return

        dados_palavra = self.data_manager.buscar_palavra(palavra)

        if dados_palavra:
            self._exibir_resultados(dados_palavra)
        else:
            self._exibir_mensagem(f"Palavra '{palavra}' não encontrada.", "error")

    def _limpar_resultados(self):
        """Limpa o widget de texto de resultados."""
        self.results_text.configure(state='normal')
        self.results_text.delete(1.0, tk.END)
        self.results_text.configure(state='disabled')

    def _inserir_texto_com_tags(self, texto, tags, nova_linha=True):
        """Insere texto no widget Text com as tags especificadas."""
        self.results_text.insert(tk.END, texto, tags)
        if nova_linha:
            self.results_text.insert(tk.END, "\n")

    def _exibir_mensagem(self, mensagem, tag_tipo):
        """Exibe uma mensagem (erro ou info) na área de resultados."""
        self._limpar_resultados()
        self.results_text.configure(state='normal')
        self._inserir_texto_com_tags(mensagem, tag_tipo, nova_linha=False)
        self.results_text.configure(state='disabled')


    def _exibir_resultados(self, dados_palavra):
        """Exibe os dados da palavra formatados no widget Text."""
        self._limpar_resultados()
        self.results_text.configure(state='normal')

        # Palavra, Part of Speech, Grammar
        self._inserir_texto_com_tags(dados_palavra.get("word", "N/A").capitalize(), "word_title")
        pos_grammar = f"{dados_palavra.get('part_of_speech', '')} {dados_palavra.get('grammar', '')}".strip()
        if pos_grammar:
            self._inserir_texto_com_tags(pos_grammar, "pos_grammar")
        
        self.results_text.insert(tk.END, "\n") # Espaço extra

        # Pronúncias
        pronunciations = dados_palavra.get("pronunciations")
        if pronunciations:
            self._inserir_texto_com_tags("Pronúncias:", "heading")
            for region, pron_data in pronunciations.items():
                self.results_text.insert(tk.END, f"  {region.upper()}: ", "pron_label")
                self.results_text.insert(tk.END, f"{pron_data.get('ipa', 'N/A')} ", "ipa")
                
                # Link de áudio (simulado)
                audio_url = pron_data.get('audio')
                if audio_url:
                    start_index = self.results_text.index(tk.INSERT)
                    self.results_text.insert(tk.END, self.AUDIO_SYMBOL, ("audio_link",))
                    end_index = self.results_text.index(tk.INSERT)
                    # Adiciona a URL como um atributo da tag para ser recuperada no evento
                    self.results_text.tag_add(f"audio_url_{audio_url}", start_index, end_index) 
                    self.results_text.tag_config(f"audio_url_{audio_url}", foreground="blue", underline=True)
                    self.results_text.tag_bind(f"audio_url_{audio_url}", "<Button-1>", 
                                               lambda e, url=audio_url: self._clique_audio_link(e, url))

                self.results_text.insert(tk.END, "\n")
            self.results_text.insert(tk.END, "\n")

        # Senses (Definições e Exemplos)
        senses = dados_palavra.get("senses")
        if senses:
            self._inserir_texto_com_tags("Definições:", "heading")
            for i, sense in enumerate(senses):
                if i > 0 : self.results_text.insert(tk.END, "-----\n", "info") # Separador entre senses
                
                cefr = sense.get("cefr_level")
                if cefr:
                    self._inserir_texto_com_tags(f"  {cefr}", "cefr_level")
                
                self._inserir_texto_com_tags(f"  {sense.get('definition', 'N/A')}", "definition")
                
                examples = sense.get("examples")
                if examples:
                    self._inserir_texto_com_tags("    Exemplos:", "example_intro", nova_linha=False)
                    self.results_text.insert(tk.END, "\n")
                    for ex in examples:
                        self._inserir_texto_com_tags(f"    • {ex}", "example_text")
                self.results_text.insert(tk.END, "\n") # Espaço após cada sense
        
        # SMART Vocabulary
        smart_vocab = dados_palavra.get("smart_vocabulary")
        if smart_vocab:
            self._inserir_texto_com_tags("SMART Vocabulary:", "heading")
            topic = smart_vocab.get("topic")
            if topic and topic.get("name"):
                self._inserir_texto_com_tags(f"  Tópico: {topic.get('name')}", "smart_topic_name")
                if topic.get("url"):
                    self.results_text.insert(tk.END, "    ")
                    start_idx = self.results_text.index(tk.INSERT)
                    self.results_text.insert(tk.END, "(ver online)", ("smart_link",))
                    end_idx = self.results_text.index(tk.INSERT)
                    self.results_text.tag_add(f"smart_url_{topic.get('url')}", start_idx, end_idx)
                    self.results_text.tag_bind(f"smart_url_{topic.get('url')}", "<Button-1>",
                                               lambda e, url=topic.get('url'): self._clique_smart_link(e, url))
                    self.results_text.insert(tk.END, "\n")

            related_words = smart_vocab.get("related_words")
            if related_words:
                self._inserir_texto_com_tags("    Palavras Relacionadas:", "smart_topic_name") # Reutilizando tag para consistência
                for rel_word_data in related_words:
                    word_text = f"      • {rel_word_data.get('word')}"
                    self.results_text.insert(tk.END, word_text, "smart_related_word")
                    url = rel_word_data.get('url')
                    if url:
                        self.results_text.insert(tk.END, " ", "smart_related_word") # Espaço antes do link
                        start_idx = self.results_text.index(tk.INSERT)
                        self.results_text.insert(tk.END, "(link)", ("smart_link",))
                        end_idx = self.results_text.index(tk.INSERT)
                        self.results_text.tag_add(f"smart_url_{url.strip()}", start_idx, end_idx) # strip() em URL
                        self.results_text.tag_bind(f"smart_url_{url.strip()}", "<Button-1>",
                                                   lambda e, u=url.strip(): self._clique_smart_link(e, u))
                    self.results_text.insert(tk.END, "\n")
            self.results_text.insert(tk.END, "\n")

        self.results_text.configure(state='disabled')
        self.results_text.see(1.0) # Rola para o topo


    def _clique_audio_link(self, event, url):
        """
        Ação ao clicar no link de áudio.
        Atualmente, abre o link no navegador.
        Para tocar o áudio diretamente, seria necessário uma biblioteca como playsound
        e, possivelmente, baixar o áudio primeiro se for um stream.
        """
        if url:
            try:
                # Tentativa simples de "tocar" abrindo no navegador ou player padrão
                # Idealmente, usar playsound(url) ou baixar e tocar.
                # Por segurança e simplicidade, vamos apenas abrir no navegador.
                if messagebox.askyesno("Abrir Áudio", f"Deseja tentar abrir o link do áudio ({url}) no seu navegador/player padrão?"):
                    webbrowser.open_new_tab(url)
            except Exception as e:
                messagebox.showerror("Erro Áudio", f"Não foi possível abrir o link de áudio: {e}")
        else:
            messagebox.showinfo("Áudio", "Nenhum link de áudio disponível.")
        return "break" # Impede que o Text widget processe mais o clique

    def _clique_smart_link(self, event, url):
        """Ação ao clicar em um link do SMART Vocabulary."""
        if url:
            try:
                if messagebox.askyesno("Abrir Link", f"Deseja abrir o link ({url}) no seu navegador?"):
                    webbrowser.open_new_tab(url)
            except Exception as e:
                messagebox.showerror("Erro Link", f"Não foi possível abrir o link: {e}")
        else:
            messagebox.showinfo("Link", "Nenhum URL disponível para este item.")
        return "break"

In [4]:
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
# from data_manager import DataManager
import webbrowser # Para abrir links e simular "tocar áudio"

class DicionarioApp:
    """
    Classe principal para a aplicação de dicionário com interface Tkinter.
    """
    AUDIO_SYMBOL = "🔊" # U+1F50A

    def __init__(self, master_root):
        """
        Inicializa a aplicação.

        Args:
            master_root (tk.Tk): A janela raiz da aplicação.
        """
        self.root = master_root
        self.root.title("Dicionário Elegante")
        self.root.geometry("800x700")

        self.data_manager = DataManager() # Carrega os dados ao instanciar

        if not self.data_manager.data:
            messagebox.showerror("Erro de Dados", 
                                 "Não foi possível carregar os dados do dicionário. "
                                 "Verifique o arquivo 'seu_arquivo_dicionario.json' e reinicie a aplicação.")
            self.root.destroy()
            return
            
        self._configurar_estilos_ttk()
        self._criar_widgets()
        self._configurar_tags_texto()

    def _configurar_estilos_ttk(self):
        """Configura estilos para widgets ttk para uma aparência mais moderna."""
        style = ttk.Style()
        style.theme_use('clam') # 'clam', 'alt', 'default', 'classic'

        style.configure("TFrame", background="#f0f0f0")
        style.configure("TLabel", background="#f0f0f0", font=("Arial", 10))
        style.configure("TButton", font=("Arial", 10, "bold"), padding=5)
        style.configure("Search.TEntry", padding=5, font=("Arial", 11))
        
        style.configure("Header.TLabel", font=("Arial", 12, "bold"), foreground="#333333")
        style.configure("WordTitle.TLabel", font=("Arial", 20, "bold"), foreground="#003366") # Exemplo, não usado diretamente no Text
        style.configure("POS.TLabel", font=("Arial", 14, "italic"), foreground="#555555") # Exemplo

    def _criar_widgets(self):
        """Cria e organiza os widgets da interface."""
        
        # Frame principal
        main_frame = ttk.Frame(self.root, padding="10 10 10 10")
        main_frame.pack(expand=True, fill=tk.BOTH)

        # --- Frame de Busca ---
        search_frame = ttk.Frame(main_frame, padding="5")
        search_frame.pack(fill=tk.X, pady=(0, 10))

        ttk.Label(search_frame, text="Digite a palavra:", font=("Arial", 11)).pack(side=tk.LEFT, padx=(0, 5))
        
        self.search_entry = ttk.Entry(search_frame, width=40, style="Search.TEntry")
        self.search_entry.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=5)
        self.search_entry.bind("<Return>", self._processar_busca_event)

        search_button = ttk.Button(search_frame, text="Buscar", command=self._processar_busca)
        search_button.pack(side=tk.LEFT, padx=(5, 0))

        # --- Frame de Resultados ---
        results_frame = ttk.Frame(main_frame, padding="5")
        results_frame.pack(expand=True, fill=tk.BOTH)
        
        # Usando scrolledtext.ScrolledText para rolagem automática
        self.results_text = scrolledtext.ScrolledText(
            results_frame, 
            wrap=tk.WORD, 
            font=("Arial", 11), 
            padx=10, pady=10,
            relief=tk.SOLID,
            bd=1,
            bg="white" # Fundo branco para a área de texto
        )
        self.results_text.pack(expand=True, fill=tk.BOTH)
        self.results_text.configure(state='disabled') # Inicia desabilitado para edição

    def _configurar_tags_texto(self):
        """Configura as tags de estilo para o widget Text."""
        self.results_text.tag_configure("word_title", font=("Arial", 22, "bold"), foreground="#003366", spacing1=5, spacing3=5)
        self.results_text.tag_configure("pos_grammar", font=("Arial", 14, "italic"), foreground="#444444", spacing3=3)
        
        self.results_text.tag_configure("heading", font=("Arial", 13, "bold", "underline"), foreground="#222222", spacing1=10, spacing3=5)
        
        self.results_text.tag_configure("pron_label", font=("Arial", 11, "bold"), foreground="#005588")
        self.results_text.tag_configure("ipa", font=("Arial", 11, "italic"), foreground="#333333")
        
        self.results_text.tag_configure("audio_link", foreground="blue", underline=True)
        self.results_text.tag_bind("audio_link", "<Button-1>", self._clique_audio_link)
        
        self.results_text.tag_configure("cefr_level", font=("Arial", 11, "bold"), foreground="#880000", spacing3=3)
        self.results_text.tag_configure("definition", font=("Arial", 11), foreground="#111111", spacing3=3, lmargin1=10, lmargin2=10)
        self.results_text.tag_configure("example_intro", font=("Arial", 10, "italic"), foreground="#555555", lmargin1=20, lmargin2=20)
        self.results_text.tag_configure("example_text", font=("Arial", 10), foreground="#333333", lmargin1=25, lmargin2=25, spacing3=2)

        self.results_text.tag_configure("smart_topic_name", font=("Arial", 11, "bold"), foreground="#006622", lmargin1=10, lmargin2=10)
        self.results_text.tag_configure("smart_related_word", font=("Arial", 10), foreground="#006622", lmargin1=20, lmargin2=20)
        self.results_text.tag_configure("smart_link", foreground="blue", underline=True, lmargin1=20, lmargin2=20)
        self.results_text.tag_bind("smart_link", "<Button-1>", self._clique_smart_link)

        self.results_text.tag_configure("error", font=("Arial", 12, "bold"), foreground="red", justify=tk.CENTER, spacing1=20)
        self.results_text.tag_configure("info", font=("Arial", 11, "italic"), foreground="gray", justify=tk.CENTER, spacing1=20)


    def _processar_busca_event(self, event=None): # Adicionado event=None
        """Processa o evento de busca (ex: Enter no Entry)."""
        self._processar_busca()

    def _processar_busca(self):
        """Obtém a palavra e exibe os resultados."""
        palavra = self.search_entry.get().strip()
        if not palavra:
            self._exibir_mensagem("Por favor, digite uma palavra.", "info")
            return

        dados_palavra = self.data_manager.buscar_palavra(palavra)

        if dados_palavra:
            self._exibir_resultados(dados_palavra)
        else:
            self._exibir_mensagem(f"Palavra '{palavra}' não encontrada.", "error")

    def _limpar_resultados(self):
        """Limpa o widget de texto de resultados."""
        self.results_text.configure(state='normal')
        self.results_text.delete(1.0, tk.END)
        self.results_text.configure(state='disabled')

    def _inserir_texto_com_tags(self, texto, tags, nova_linha=True):
        """Insere texto no widget Text com as tags especificadas."""
        self.results_text.insert(tk.END, texto, tags)
        if nova_linha:
            self.results_text.insert(tk.END, "\n")

    def _exibir_mensagem(self, mensagem, tag_tipo):
        """Exibe uma mensagem (erro ou info) na área de resultados."""
        self._limpar_resultados()
        self.results_text.configure(state='normal')
        self._inserir_texto_com_tags(mensagem, tag_tipo, nova_linha=False)
        self.results_text.configure(state='disabled')


    def _exibir_resultados(self, dados_palavra):
        """Exibe os dados da palavra formatados no widget Text."""
        self._limpar_resultados()
        self.results_text.configure(state='normal')

        # Palavra, Part of Speech, Grammar
        self._inserir_texto_com_tags(dados_palavra.get("word", "N/A").capitalize(), "word_title")
        pos_grammar = f"{dados_palavra.get('part_of_speech', '')} {dados_palavra.get('grammar', '')}".strip()
        if pos_grammar:
            self._inserir_texto_com_tags(pos_grammar, "pos_grammar")
        
        self.results_text.insert(tk.END, "\n") # Espaço extra

        # Pronúncias
        pronunciations = dados_palavra.get("pronunciations")
        if pronunciations:
            self._inserir_texto_com_tags("Pronúncias:", "heading")
            for region, pron_data in pronunciations.items():
                self.results_text.insert(tk.END, f"  {region.upper()}: ", "pron_label")
                self.results_text.insert(tk.END, f"{pron_data.get('ipa', 'N/A')} ", "ipa")
                
                # Link de áudio (simulado)
                audio_url = pron_data.get('audio')
                if audio_url:
                    start_index = self.results_text.index(tk.INSERT)
                    self.results_text.insert(tk.END, self.AUDIO_SYMBOL, ("audio_link",))
                    end_index = self.results_text.index(tk.INSERT)
                    # Adiciona a URL como um atributo da tag para ser recuperada no evento
                    self.results_text.tag_add(f"audio_url_{audio_url}", start_index, end_index) 
                    self.results_text.tag_config(f"audio_url_{audio_url}", foreground="blue", underline=True)
                    self.results_text.tag_bind(f"audio_url_{audio_url}", "<Button-1>", 
                                               lambda e, url=audio_url: self._clique_audio_link(e, url))

                self.results_text.insert(tk.END, "\n")
            self.results_text.insert(tk.END, "\n")

        # Senses (Definições e Exemplos)
        senses = dados_palavra.get("senses")
        if senses:
            self._inserir_texto_com_tags("Definições:", "heading")
            for i, sense in enumerate(senses):
                if i > 0 : self.results_text.insert(tk.END, "-----\n", "info") # Separador entre senses
                
                cefr = sense.get("cefr_level")
                if cefr:
                    self._inserir_texto_com_tags(f"  {cefr}", "cefr_level")
                
                self._inserir_texto_com_tags(f"  {sense.get('definition', 'N/A')}", "definition")
                
                examples = sense.get("examples")
                if examples:
                    self._inserir_texto_com_tags("    Exemplos:", "example_intro", nova_linha=False)
                    self.results_text.insert(tk.END, "\n")
                    for ex in examples:
                        self._inserir_texto_com_tags(f"    • {ex}", "example_text")
                self.results_text.insert(tk.END, "\n") # Espaço após cada sense
        
        # SMART Vocabulary
        smart_vocab = dados_palavra.get("smart_vocabulary")
        if smart_vocab:
            self._inserir_texto_com_tags("SMART Vocabulary:", "heading")
            topic = smart_vocab.get("topic")
            if topic and topic.get("name"):
                self._inserir_texto_com_tags(f"  Tópico: {topic.get('name')}", "smart_topic_name")
                if topic.get("url"):
                    self.results_text.insert(tk.END, "    ")
                    start_idx = self.results_text.index(tk.INSERT)
                    self.results_text.insert(tk.END, "(ver online)", ("smart_link",))
                    end_idx = self.results_text.index(tk.INSERT)
                    self.results_text.tag_add(f"smart_url_{topic.get('url')}", start_idx, end_idx)
                    self.results_text.tag_bind(f"smart_url_{topic.get('url')}", "<Button-1>",
                                               lambda e, url=topic.get('url'): self._clique_smart_link(e, url))
                    self.results_text.insert(tk.END, "\n")

            related_words = smart_vocab.get("related_words")
            if related_words:
                self._inserir_texto_com_tags("    Palavras Relacionadas:", "smart_topic_name") # Reutilizando tag para consistência
                for rel_word_data in related_words:
                    word_text = f"      • {rel_word_data.get('word')}"
                    self.results_text.insert(tk.END, word_text, "smart_related_word")
                    url = rel_word_data.get('url')
                    if url:
                        self.results_text.insert(tk.END, " ", "smart_related_word") # Espaço antes do link
                        start_idx = self.results_text.index(tk.INSERT)
                        self.results_text.insert(tk.END, "(link)", ("smart_link",))
                        end_idx = self.results_text.index(tk.INSERT)
                        self.results_text.tag_add(f"smart_url_{url.strip()}", start_idx, end_idx) # strip() em URL
                        self.results_text.tag_bind(f"smart_url_{url.strip()}", "<Button-1>",
                                                   lambda e, u=url.strip(): self._clique_smart_link(e, u))
                    self.results_text.insert(tk.END, "\n")
            self.results_text.insert(tk.END, "\n")

        self.results_text.configure(state='disabled')
        self.results_text.see(1.0) # Rola para o topo


    def _clique_audio_link(self, event, url):
        """
        Ação ao clicar no link de áudio.
        Atualmente, abre o link no navegador.
        Para tocar o áudio diretamente, seria necessário uma biblioteca como playsound
        e, possivelmente, baixar o áudio primeiro se for um stream.
        """
        if url:
            try:
                # Tentativa simples de "tocar" abrindo no navegador ou player padrão
                # Idealmente, usar playsound(url) ou baixar e tocar.
                # Por segurança e simplicidade, vamos apenas abrir no navegador.
                if messagebox.askyesno("Abrir Áudio", f"Deseja tentar abrir o link do áudio ({url}) no seu navegador/player padrão?"):
                    webbrowser.open_new_tab(url)
            except Exception as e:
                messagebox.showerror("Erro Áudio", f"Não foi possível abrir o link de áudio: {e}")
        else:
            messagebox.showinfo("Áudio", "Nenhum link de áudio disponível.")
        return "break" # Impede que o Text widget processe mais o clique

    def _clique_smart_link(self, event, url):
        """Ação ao clicar em um link do SMART Vocabulary."""
        if url:
            try:
                if messagebox.askyesno("Abrir Link", f"Deseja abrir o link ({url}) no seu navegador?"):
                    webbrowser.open_new_tab(url)
            except Exception as e:
                messagebox.showerror("Erro Link", f"Não foi possível abrir o link: {e}")
        else:
            messagebox.showinfo("Link", "Nenhum URL disponível para este item.")
        return "break"

In [5]:
import tkinter as tk
# from app_gui import DicionarioApp

if __name__ == '__main__':
    root = tk.Tk()
    app = DicionarioApp(root)
    # A verificação de dados e destruição da root é feita em DicionarioApp.__init__
    # Então, só iniciamos o mainloop se a root ainda existir.
    if root.winfo_exists(): 
        root.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
TypeError: DicionarioApp._clique_smart_link() missing 1 required positional argument: 'url'
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
TypeError: DicionarioApp._clique_audio_link() missing 1 required positional argument: 'url'
